# What was renovated? Renovated parts and energy measures from the listing descriptions

**This Jupyter notebook does the following:**

- Imports the cleaned apartment data `Data/apartments_pre1990_clean.csv` (from `04_combine_and_clean.ipynb`)
- Finds in the listing description **which parts** of the apartment were renovated (kitchen, bathroom, floors, paint, complete renovation)
- Finds **which energy measures** are mentioned (windows, heating, insulation/facade, roof, solar, Minergie)
- Flags listings that are probably **new buildings** (they should not be in a data set of buildings built before 1990)
- Saves the data with the new columns to `Data/apartments_pre1990_renovation_details.csv`

The descriptions are the texts of the original listings on flatfox and ImmoScout24 (scraped in week 7). The renovation groups are **not** changed; the new columns only describe them in more detail.

## Libraries and settings

In [1]:
# Libraries
import os
import re
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

c:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Importing data

In [2]:
df = pd.read_csv('./Data/apartments_pre1990_clean.csv', sep=',', encoding='utf-8')

print('Dimension:', df.shape, '\n')
print(df['renovation_group'].value_counts())

Dimension: (733, 22) 

renovation_group
general renovation    486
not renovated         201
energy renovation      46
Name: count, dtype: int64


## Search patterns

The descriptions are in German, French, Italian and English, so all patterns contain words in these languages. All texts are converted to lower case first.

**Rule for the renovated parts and the windows/heating/roof:** a part counts as renovated if it appears in the **same sentence** as a renovation word (e.g. *neu*, *erneuert*, *renoviert*, *saniert*, *rénové*, *rinnovato*). Example: *"Bad und Küche wurden 2016 komplett renoviert"* → bathroom and kitchen.

**Rule for energy technologies:** some words describe an energy measure by themselves and count even without a renovation word, e.g. *Wärmepumpe*, *Photovoltaik*, *3-fach verglast*, *Dämmung*, *Minergie*.

Phrases like *"Ihr neues Zuhause"* ("your new home") are removed first, because *neu* there does not mean a renovation. `` is a word boundary, so that *neu* does not match *Neuchâtel* or *Neuhaus*.

In [3]:
# Renovation words (one of them must be in the same sentence as the part)
reno_words = (r'(\bneu(e|en|er|es|em)?\b|erneuert|renoviert|saniert|ersetzt|modernisiert|umgebaut'
              r'|rénov|refait|\bneu(f|ve|ves|fs)\b|\bnouve|remplac'
              r'|rinnovat|ristrutturat|\bnuov|sostituit'
              r'|renovated|\bnew\b)')

# Phrases with 'new' that do not mean a renovation (removed before the search)
not_reno = r'(neu(es|e|en) zuhause|neue(n)? mieter|nouveau chez|nouvelle maison|nuova casa|new home)'

# Complete renovation (the whole apartment or building)
complete_words = (r'(total ?san|total ?renov|kernsan|komplett (renoviert|saniert|erneuert)|vollständig (renoviert|saniert)'
                  r'|gesamtsan|umfassend (renoviert|saniert)|erstbezug nach|frisch (renoviert|saniert)|neu renoviert'
                  r'|entièrement rénov|complètement rénov|totalement rénov'
                  r'|completamente (rinnovat|ristrutturat)|fully renovated|completely renovated)')

# Parts of the apartment (renovated if in the same sentence as a renovation word)
part_words = {
    'kitchen':  r'(küche|cuisine|cucina|kitchen)',
    'bathroom': r'(\bbad\b|badezimmer|nasszelle|dusche|sanitär|salle de bain|salle de douche|bagno|bathroom)',
    'floors':   r'(b(o|ö)den(?!see)|parkett|laminat|\bsols?\b|parquet|paviment|floor)',
    'paint':    r'(gestrichen|streichen|malerarbeiten|repeint|peinture|tinteggiat|painted)',
}

# Energy measures (renovated if in the same sentence as a renovation word ...)
energy_words = {
    'windows':    r'(fenster|fenêtre|vitrage|finestr|serrament|window)',
    'heating':    r'(heizung|chauffage|riscaldamento|heating)',
    'insulation': r'(fassade|gebäudehülle|dämmung|gedämmt|isoliert|façade|facciat|cappotto|isolat|insulat)',
    'roof':       r'(\bdach\b|toiture|\btoit\b|\btetto\b|\broof\b)',
}

# ... or always, if these words are mentioned
energy_always = {
    'windows':    r'(3-fach|3fach|dreifach|triple vitrage|tripli vetri|triplo vetro)',
    'heating':    r'(wärmepumpe|erdsonde|pellet|pompe à chaleur|pompa di calore|heat pump)',
    'insulation': r'(dämmung|gedämmt|gebäudehülle|energetisch saniert|isolation thermique|cappotto)',
    'solar':      r'(photovoltaik|solaranlage|photovoltaï|panneaux solaires|fotovoltaic|pannelli solari|solar panel)',
    'minergie':   r'(minergie)',
}

# Probably a new building (should not be in a data set of buildings built before 1990)
new_build_words = (r'(neubau(projekt)? 20|neubauprojekt|neubauwohnung|neubau!|vermietung ab plan'
                   r'|appartements? neufs|nouvelle construction|immeuble neuf|nuova costruzione|nuovo stabile|new build)')

## Search the descriptions

The function `find_parts()` splits a description into sentences and returns the set of parts and energy measures that were found.

In [4]:
def split_sentences(text):
    return [s for s in re.split(r'[.!?\n;•]|\s-\s|:\s', text) if s.strip()]

def find_parts(text):
    text = '' if not isinstance(text, str) else re.sub(not_reno, '', text.lower())
    found = set()

    # Parts and energy measures in a sentence with a renovation word
    for s in split_sentences(text):
        if re.search(reno_words, s):
            for name, pattern in {**part_words, **energy_words}.items():
                if re.search(pattern, s):
                    found.add(name)

    # Energy technologies (always)
    for name, pattern in energy_always.items():
        if re.search(pattern, text):
            found.add(name)

    # Complete renovation
    if re.search(complete_words, text):
        found.add('complete')
    return found

parts = df['description'].map(find_parts)

# One True/False column per part
part_cols = ['complete', 'kitchen', 'bathroom', 'floors', 'paint']
energy_cols = ['windows', 'heating', 'insulation', 'roof', 'solar', 'minergie']

for c in part_cols:
    df['reno_' + c] = parts.map(lambda s: c in s)
for c in energy_cols:
    df['energy_' + c] = parts.map(lambda s: c in s)

# Minergie also from the column 'minergie' (flatfox attribute / ImmoScout24 listing card)
df['energy_minergie'] = df['energy_minergie'] | df['minergie'].astype(bool)

# Text column with all parts found, e.g. 'kitchen, bathroom, windows'
def parts_text(row):
    names = [c for c in part_cols if row['reno_' + c]] + [c for c in energy_cols if row['energy_' + c]]
    if names:
        return ', '.join(names)
    return 'none' if row['renovation_group'] == 'not renovated' else 'not specified'

df['renovated_parts'] = df.apply(parts_text, axis=1)

# Possible new buildings
df['possible_new_build'] = df['description'].fillna('').str.lower().str.contains(new_build_words, regex=True)

df[['address', 'renovation_group', 'renovated_parts']].head(10)

,address,renovation_group,renovated_parts
0,"Croix Fédérale 44, 2300 La Chaux-de-Fonds",general renovation,not specified
1,"Grüttbachstr. 6, 4542 Luterbach",general renovation,not specified
2,"Neuhofstr. 20, 5600 Lenzburg",energy renovation,"kitchen, bathroom, windows"
3,"Aathalstr. 19, 8610 Uster",not renovated,none
4,"Rte de Bertigny 2/4, 1700 Fribourg",general renovation,not specified
5,"Aeussere Altmatt 19, 6418 Rothenthurm",general renovation,not specified
6,"Carl Spitteler - Str. 30, 8053 Zürich",general renovation,not specified
7,"Steinentorstrasse 14, 4051 Basel",general renovation,not specified
8,"Thurgauerstr. 31, 8400 Winterthur",general renovation,not specified
9,"Holzgasse 27, 4537 Wiedlisbach",energy renovation,"complete, heating"


## Results

### Renovated parts and energy measures by renovation group

Number of apartments in each group that mention a part (one apartment can have several parts).

In [5]:
cols = ['reno_' + c for c in part_cols] + ['energy_' + c for c in energy_cols]

summary = df.groupby('renovation_group')[cols].sum().T
summary['total'] = summary.sum(axis=1)
summary.loc['number of apartments'] = df['renovation_group'].value_counts()
summary.loc['number of apartments', 'total'] = len(df)
summary = summary[['energy renovation', 'general renovation', 'not renovated', 'total']].astype(int)
summary

renovation_group,energy renovation,general renovation,not renovated,total
reno_complete,23,132,0,155
reno_kitchen,10,72,2,84
reno_bathroom,10,56,0,66
reno_floors,6,30,2,38
reno_paint,1,14,3,18
energy_windows,15,8,1,24
energy_heating,17,2,0,19
energy_insulation,8,1,0,9
energy_roof,2,1,0,3
energy_solar,12,0,0,12


In [6]:
# Apartments where the description does not say what was renovated
for g in ['energy renovation', 'general renovation']:
    sub = df[df['renovation_group'] == g]
    k = (sub['renovated_parts'] == 'not specified').sum()
    print(f'{g}: {k} of {len(sub)} without details ({k / len(sub) * 100:.0f} %)')

energy renovation: 1 of 46 without details (2 %)
general renovation: 309 of 486 without details (64 %)


### Energy renovation: what was done?

All apartments of the group *energy renovation* with the parts and energy measures that were found. Review this list: it is the treatment group of Topic 1.

In [7]:
pd.set_option('display.max_colwidth', 80)
df.loc[df['renovation_group'] == 'energy renovation',
       ['source', 'address', 'year_built_clean', 'year_renovated_clean', 'renovated_parts', 'possible_new_build']]

,source,address,year_built_clean,year_renovated_clean,renovated_parts,possible_new_build
2,immoscout24,"Neuhofstr. 20, 5600 Lenzburg",1960.0,NaN,"kitchen, bathroom, windows",False
9,immoscout24,"Holzgasse 27, 4537 Wiedlisbach",1962.0,NaN,"complete, heating",False
43,both,"Ch. des Sources 16, 2013 Colombier NE",1988.0,2024.0,"complete, kitchen, solar, minergie",False
52,both,"Via Franco Zorzi 25a, 6500 Bellinzona",1985.0,NaN,solar,False
55,immoscout24,"Ch. des Sources 16, 2013 Colombier NE",1988.0,2024.0,"complete, kitchen, solar, minergie",False
58,immoscout24,"Hohlstrasse 90, 8004 Zürich",1891.0,2024.0,"complete, minergie",False
71,immoscout24,"Ch. des Sources 16a, 2013 Colombier NE",1988.0,2024.0,"complete, kitchen, solar, minergie",False
109,immoscout24,"Rütistrasse 29, 8702 Zollikon",1911.0,NaN,windows,False
162,immoscout24,"Rickenstrasse 12, 8732 Neuhaus",1865.0,2026.0,"complete, bathroom, heating, insulation",False
166,immoscout24,"Rathausgasse 24, 3011 Bern",1919.0,1989.0,"complete, bathroom, floors, windows, heating",False


### General renovation: what was done?

Most frequent combinations of renovated parts in the group *general renovation*.

In [8]:
df.loc[df['renovation_group'] == 'general renovation', 'renovated_parts'].value_counts().head(15)

renovated_parts
not specified                                   309
complete                                         76
kitchen, bathroom                                14
complete, kitchen                                13
kitchen                                           9
complete, kitchen, bathroom                       9
bathroom                                          7
complete, kitchen, bathroom, floors               5
complete, bathroom                                5
minergie                                          4
complete, kitchen, bathroom, floors, paint        4
kitchen, bathroom, floors                         3
complete, kitchen, floors                         2
complete, kitchen, bathroom, floors, windows      2
complete, floors                                  2
Name: count, dtype: int64

### Possible new buildings

These listings say "Neubau", "appartements neufs" or similar, but have a construction year before 1990 or no construction year. They are probably new buildings with a wrong construction year on the platform, or new flats in an old building. They are **flagged, not removed** — check them on the platform and decide in the team.

In [9]:
df.loc[df['possible_new_build'], ['source', 'address', 'year_built_clean', 'renovation_group', 'listing_url']]

,source,address,year_built_clean,renovation_group,listing_url
12,immoscout24,"Seidenhofstr. 1, 6003 Luzern",1940.0,not renovated,https://www.immoscout24.ch/mieten/4002168991
81,immoscout24,"J. J. Balmer - Str. 1a, 4053 Basel",1969.0,not renovated,https://www.immoscout24.ch/mieten/4003363338
114,immoscout24,"Seidenhofstr. 1, 6003 Luzern",1940.0,not renovated,https://www.immoscout24.ch/mieten/4003654810
115,immoscout24,"Seidenhofstr. 1, 6003 Luzern",1940.0,not renovated,https://www.immoscout24.ch/mieten/4003654815
249,immoscout24,"Route du Nord 4, 1723 Marly",1988.0,energy renovation,https://www.immoscout24.ch/mieten/4003582991
435,flatfox,1741 Cottens FR,NaN,energy renovation,https://flatfox.ch/en/flat/rte-de-la-goille-28-1741-cottens-fr/86460768/
440,flatfox,1741 Cottens FR,NaN,energy renovation,https://flatfox.ch/en/flat/rte-de-la-goille-28-1741-cottens-fr/86469440/


### Limitations

- The search uses keywords, so it can miss parts (e.g. unusual wording) or find parts that were not renovated (e.g. *"Fussbodenheizung, neue Küche"* also finds *heating*).
- Many descriptions only say *"renoviert"* or give a renovation year without saying what was done (*not specified*).
- A heat pump or solar panels can also be part of the original building; the text does not always say when they were installed.

### Save data to file

In [10]:
df.to_csv('./Data/apartments_pre1990_renovation_details.csv',
          sep=",",
          encoding='utf-8',
          index=False)

print('Saved:', df.shape[0], 'apartments,', df.shape[1], 'columns')

Saved: 733 apartments, 35 columns


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [11]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:45:05
Python Version: 3.11.8
-----------------------------------
